In [1]:
print("hi")

hi


In [ ]:
import json
import re
import os
from PIL import Image
import pdfplumber
import torch
import cv2
import numpy as np
# from pipe_fn import pipe
from output_utils import save_split_output

from transformers import pipeline

from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,
    calculate_model_confidence,
)





# =========================
# CROP TABLES
# =========================
def crop_claim_tables(pdf_path, output_dir="Marpai_output"):
    """
    Marpai Health EOPs mark the start of a claim table with 'Claim#:'
    (NO space) and the end with the 'Column Totals' row.

    Unlike Argus/Wellnet, there is no 'Provider:' block ABOVE the Claim#
    line to worry about — 'Claim#:' is the very first line of the header
    block (Claim#: / Provider Name: on the same row), so only a small top
    margin is needed.

    Below 'Column Totals' sits a single boxed line:
        Patient Responsibility: $xxx.xx
    followed (further down, OUTSIDE our crop) by 'Remark Code Messages'.
    A margin of ~26pt below 'Column Totals' captures the Patient
    Responsibility box without pulling in the Remark Code Messages
    section.
    """
    os.makedirs(output_dir, exist_ok=True)
    cropped_images = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            print(f"\n📄 Processing Page {page_num}")

            claim_hits = page.search("Claim#:")
            if not claim_hits:
                claim_hits = page.search("Claim #:")  # fallback in case spacing varies

            total_hits = page.search("Column Totals")

            if not claim_hits or not total_hits:
                continue

            table_count = min(len(claim_hits), len(total_hits))

            for idx in range(table_count):
                start_y = claim_hits[idx]["top"] - 5
                end_y   = total_hits[idx]["bottom"] + 26  # room for the
                                                           # Patient Responsibility box below

                if start_y >= end_y:
                    continue

                bbox = (0, max(start_y, 0), page.width, end_y)
                cropped_page = page.crop(bbox)

                image_path = os.path.join(
                    output_dir,
                    f"page_{page_num}_table_{idx+1}.png"
                )
                cropped_page.to_image(resolution=500).save(image_path)
                print(f"✅ Saved: {image_path}")

                expected_rows = count_service_rows(page, start_y, end_y)

                cropped_images.append({
                    "page":          page_num,
                    "table":         idx + 1,
                    "image_path":    image_path,
                    "expected_rows": expected_rows,
                })

    return cropped_images


def count_service_rows(page, region_top, region_bottom):
    """
    Marpai rows start with a date RANGE, e.g. '08/06-08/06/2025'
    (like Wellnet, NOT a single date like Argus). We count unique
    y-positions matching that pattern inside the cropped region as a
    proxy for row count.
    """
    words = page.extract_words()
    line_text = {}
    for w in words:
        y = round(float(w["top"]), 1)
        if region_top <= y <= region_bottom:
            line_text.setdefault(y, []).append(w["text"])

    date_row_pattern = re.compile(r"^\d{2}/\d{2}-\d{2}/\d{2}/\d{4}$")
    service_rows = set()
    for y, words_on_line in line_text.items():
        for tok in words_on_line:
            if date_row_pattern.match(tok):
                service_rows.add(y)
                break

    return len(service_rows)


# =========================
# TABLE ENHANCEMENT
# =========================
def make_table(image_path):
    img  = cv2.imread(image_path)
    gray = cv2.imread(image_path, 0)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
    sums  = np.sum(thresh, axis=1)
    th    = (thresh.shape[1] * 255) * 0.6
    lines = np.where(sums > th)[0]
    for l in lines:
        cv2.line(img, (0, l), (thresh.shape[1], l), (0, 0, 0), 1)
    return img


AMOUNT_FIELDS = {
    "total_charge", "not_covered", "disc_or_excluded", "eligible_expenses",
    "deduct_applied", "copay_amount", "coins_applied", "plan_payment",
    "patient_responsibility",
}


def convert_amounts_to_string(obj):
    if isinstance(obj, dict):
        new_obj = {}
        for k, v in obj.items():
            if k in AMOUNT_FIELDS:
                try:
                    new_obj[k] = f"{float(str(v).replace('$', '').replace(',', '').strip()):.2f}"
                except Exception:
                    new_obj[k] = ""
            else:
                new_obj[k] = convert_amounts_to_string(v)
        return new_obj
    elif isinstance(obj, list):
        return [convert_amounts_to_string(i) for i in obj]
    else:
        return obj


# =========================
# DATE NORMALIZATION
# =========================
def normalize_service_date(raw_date: str) -> str:
    """
    Marpai shows Dates of Service as a range, e.g. '08/06-08/06/2025'.
    We only want the single date with the full year, e.g. '08/06/2025'.
    This takes the LAST mm/dd/yyyy occurrence in the string (the end
    date, which is the only piece that carries the year).
    """
    if not raw_date:
        return ""
    matches = re.findall(r"\d{2}/\d{2}/\d{4}", raw_date)
    if matches:
        return matches[-1]
    return raw_date.strip()


# =========================
# PROMPT — uses pdf_name as eob_id
# =========================
def build_prompt(pdf_name: str) -> str:
    return f"""
Extract structured data from this Marpai Health Explanation of Payments (EOP) claim table.

STRICT RULES

1. Extract ONLY the following header fields, found above the service table:
   - patient_number      (value after "Patient #:")

2. Extract ALL service rows exactly as shown in the table body.

3. Preserve row order.

4. Do NOT skip duplicate rows (e.g. same Service Description on two
   different lines are two separate rows).

5. Read the table strictly LEFT to RIGHT.

6. Every service object must correspond to ONE visible table row.

7. Never merge two rows.

8. Never create rows that do not exist.

9. Never use the "Column Totals" row values inside service rows.

10. Stop reading service rows when the row labeled "Column Totals" is reached.

11. Extract the "Column Totals" row separately into the "totals" object.

12. Extract "Patient Responsibility" (found on the line below the totals
    row, inside a boxed/highlighted cell) into the "totals" object as
    patient_responsibility.

13. For "service_date": the table shows a DATE RANGE such as
    "08/06-08/06/2025". Return ONLY the single end date WITH the year,
    formatted as MM/DD/YYYY (e.g. "08/06/2025"). Do NOT return the full
    range.

14. Ignore: the "Remark Code Messages" table, the "Statement Totals"
    table, and the "Payment Details" section at the bottom of the page —
    do not extract their contents as service rows or as totals.

15. Money values: remove "$" and commas, keep decimals, return as strings.
    Example: "$1,234.00" -> "1234.00"

16. If a field is blank in the table return "".

17. Never infer missing values.

18. Return ONLY valid JSON — no markdown, no explanation, no comments.

COLUMN MAPPING (service rows)

service_date          <- Dates of Service (end date only, see rule 13)
service_description   <- Service Description
total_charge          <- Total Charge
not_covered            <- Not Covered
disc_or_excluded       <- Disc or Excluded
eligible_expenses      <- Eligible Expenses
deduct_applied         <- Deduct. Applied
copay_amount           <- Co-Pay Amount
paid_at                <- Balance Paid At
coins_applied          <- Coins. Applied
plan_payment           <- Plan Payment
OUTPUT JSON SCHEMA

{{

  "claimant": {{
    "value": "",
    "confidence": 0.0
  }},

  "provider_name": {{
    "value": "",
    "confidence": 0.0
  }},

  "services": [
    {{

      "service_date": {{
        "value": "",
        "confidence": 0.0
      }},

      "service_description": {{
        "value": "",
        "confidence": 0.0
      }},

      "total_charge": {{
        "value": "",
        "confidence": 0.0
      }},

      "not_covered": {{
        "value": "",
        "confidence": 0.0
      }},

      "disc_or_excluded": {{
        "value": "",
        "confidence": 0.0
      }},

      "eligible_expenses": {{
        "value": "",
        "confidence": 0.0
      }},

      "deduct_applied": {{
        "value": "",
        "confidence": 0.0
      }},

      "copay_amount": {{
        "value": "",
        "confidence": 0.0
      }},

      "coins_applied": {{
        "value": "",
        "confidence": 0.0
      }},

      "plan_payment": {{
        "value": "",
        "confidence": 0.0
      }}

    }}
  ],

  "totals": {{

    "total_charge": {{
      "value": "",
      "confidence": 0.0
    }},

    "not_covered": {{
      "value": "",
      "confidence": 0.0
    }},

    "disc_or_excluded": {{
      "value": "",
      "confidence": 0.0
    }},

    "eligible_expenses": {{
      "value": "",
      "confidence": 0.0
    }},

    "deduct_applied": {{
      "value": "",
      "confidence": 0.0
    }},

    "copay_amount": {{
      "value": "",
      "confidence": 0.0
    }},

    "coins_applied": {{
      "value": "",
      "confidence": 0.0
    }},

    "plan_payment": {{
      "value": "",
      "confidence": 0.0
    }},

    "patient_responsibility": {{
      "value": "",
      "confidence": 0.0
    }}

  }}

}}


 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{{
  "value": "",
  "confidence": ""
}}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.

VALIDATION RULES
Number of service objects must equal the number of visible service rows.
Duplicate service descriptions must be extracted as separate rows.
Do not include the "Column Totals" row inside services.
Output ONLY JSON.
"""


# =========================
# AMOUNT HELPERS
# =========================
def parse_amount(x) -> float:
    if x is None or str(x).strip() == "":
        return 0.0
    try:
        return float(str(x).replace("$", "").replace(",", "").replace("%", "").strip())
    except ValueError:
        return 0.0


# Fields summed from service rows to check against the "Column Totals" row.
# NOTE: service_description, paid_at (a percentage, not a dollar amount),
# and remark_code are not summable, so they're excluded from totals
# validation.
TOTALS_FIELDS = [
    "total_charge", "not_covered", "disc_or_excluded", "eligible_expenses",
    "deduct_applied", "copay_amount", "coins_applied", "plan_payment",
]


def compute_totals_from_services(services: list) -> dict:
    return {
        f: round(sum(parse_amount(s.get(f, "")) for s in services), 2)
        for f in TOTALS_FIELDS
    }


def services_are_empty(services: list) -> bool:
    """Return True when every amount field in every service row is blank/zero."""
    for svc in services:
        for f in TOTALS_FIELDS:
            if str(svc.get(f, "")).strip() not in ("", "0.00", "0"):
                return False
    return True


# =========================
# VALIDATION
# =========================
def validate_patient_totals(patient: dict, patient_name: str, expected_row_count: int):
    services = patient.get("services", [])
    totals   = patient.get("totals", {})

    if not services:
        return False, "No services found", [{"error": "empty services"}]

    if services_are_empty(services):
        msg = "All service rows are empty — model likely failed to extract data"
        print(f"\n❌ {msg}")
        return False, msg, [{"error": "all_service_rows_empty"}]

    computed_totals = compute_totals_from_services(services)
    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [{patient_name}]")
    print("-" * 80)

    for field, computed_value in computed_totals.items():
        extracted_value = round(parse_amount(totals.get(field, "")), 2)
        diff  = round(computed_value - extracted_value, 2)
        match = abs(diff) <= 0.01
        icon   = "✅" if match else "❌"
        status = "MATCH" if match else "MISMATCH"

        if not match:
            has_error = True
            errors.append({
                "type":       "field_mismatch",
                "field":      field,
                "computed":   computed_value,
                "extracted":  extracted_value,
                "difference": diff,
            })

        line = (
            f"{icon} {field:25s} computed={computed_value:<10} "
            f"| extracted={extracted_value:<10} {status}"
        )
        print(line)
        result_validation += "\n" + line

    # Cross-check: Patient Responsibility = Total Charge - Disc/Excluded
    # - Plan Payment. (Not Covered and Deductible/Co-Pay/Coinsurance are
    # NOT separately subtracted here -- they are exactly the amounts that
    # flow through to the patient, so they're implicitly included in this
    # difference. Verified against the sample EOP: $653.00 - $0.00 -
    # $494.07 = $158.93, matching the extracted Patient Responsibility.)
    total_charge_sum   = computed_totals.get("total_charge", 0.0)
    disc_excluded_sum  = computed_totals.get("disc_or_excluded", 0.0)
    plan_payment_sum   = computed_totals.get("plan_payment", 0.0)
    patient_resp_extracted = round(parse_amount(totals.get("patient_responsibility", "")), 2)
    expected_patient_resp  = round(total_charge_sum - disc_excluded_sum - plan_payment_sum, 2)
    resp_match = abs(expected_patient_resp - patient_resp_extracted) <= 0.01
    icon   = "✅" if resp_match else "❌"
    status = "MATCH" if resp_match else "MISMATCH"
    if not resp_match:
        has_error = True
        errors.append({
            "type":       "field_mismatch",
            "field":      "patient_responsibility_crosscheck",
            "computed":   expected_patient_resp,
            "extracted":  patient_resp_extracted,
            "difference": round(expected_patient_resp - patient_resp_extracted, 2),
        })
    line = (
        f"{icon} {'patient_responsibility_crosscheck':25s} computed={expected_patient_resp:<10} "
        f"| extracted={patient_resp_extracted:<10} {status}"
    )
    print(line)
    result_validation += "\n" + line

    extracted_row_count = len(services)
    match  = expected_row_count == extracted_row_count
    icon   = "✅" if match else "❌"
    status = "MATCH" if match else "MISMATCH"

    if not match:
        has_error = True
        errors.append({
            "type":           "row_count_mismatch",
            "expected_rows":  expected_row_count,
            "extracted_rows": extracted_row_count,
        })

    line = (
        f"{icon} {'total_record_rows':25s} computed={expected_row_count:<10} "
        f"| extracted={extracted_row_count:<10} {status}"
    )
    print(line)
    result_validation += "\n" + line
    print("-" * 80)

    if has_error:
        print(f"❌ [{patient_name}] Validation FAILED\n")
        return False, result_validation, errors
    else:
        print(f"✅ [{patient_name}] Validation PASSED\n")
        return True, result_validation, []


# =========================
# JSON CLEANER
# =========================
def extract_json(text: str) -> dict:
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError("No JSON object found in model output")
    return json.loads(text[start:end])


def save_json(data, output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


# =========================
# DENIAL CHECK
# =========================
def check_claim_denied(pdf_path: str) -> str:
    """
    CONFIRMED via testing against the sample EOP: Marpai's page 2
    "Additional Information" section is generic appeal-rights boilerplate
    that appears on EVERY Marpai EOP (denied or not) and always contains
    words like "denied"/"deny"/"denial" ("...our decision to deny you a
    service or coverage", "claim denial", etc.), even when the claim was
    paid in full. Scanning all pages like the Argus/Allied/Wellnet
    scripts do produces a false positive here.

    Fix: only scan the claim table page(s) -- i.e. pages that contain
    'Claim#:' -- for denial keywords, and skip the generic boilerplate
    pages entirely.
    """
    denial_keywords = ["denied", "denial"]

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # Only inspect pages that actually contain a claim block;
            # skip generic "Additional Information" / appeal-rights pages.
            if not page.search("Claim#:") and not page.search("Claim #:"):
                continue

            full_text = page.extract_text()
            if not full_text:
                continue
            searchable = full_text.lower()
            for keyword in denial_keywords:
                if keyword in searchable:
                    print(f"claim denied keyword found: {keyword}")
                    return "denied"

    return "not denied"


# =========================
# RETRY HELPER
# =========================
MAX_RETRIES = 2

def run_model_with_retry(image: Image.Image, prompt: str, expected_rows: int):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text":  prompt},
            ],
        }
    ]

    last_parsed = None
    last_model_confidence = None

    for attempt in range(1, MAX_RETRIES + 1):
        print(f"  🔄 Model attempt {attempt}/{MAX_RETRIES}")

        with torch.no_grad():
            output = pipe(
                messages,
                max_new_tokens=1500,
                temperature=0.0,
                do_sample=False,
            )

        raw = output[0]["generated_text"]
        if isinstance(raw, list):
            raw = raw[-1]["content"]

        try:
            raw_parsed = extract_json(raw)
        except Exception as e:
            print(f"  ❌ JSON parse failed on attempt {attempt}: {e}")
            continue

        # Compute per-field confidence off the raw {value, confidence}
        # structure BEFORE unwrapping, then flatten to plain values so
        # every downstream step (amount conversion, date normalization,
        # totals validation) works on plain strings instead of
        # {"value": ..., "confidence": ...} dicts.
        model_confidence = calculate_model_confidence(raw_parsed)
        parsed = _unwrap_vlm_output(raw_parsed)

        services = parsed.get("services", [])
        row_ok   = (len(services) == expected_rows)
        empty_ok = not services_are_empty(services)

        if row_ok and empty_ok:
            print(f"  ✅ Accepted on attempt {attempt}")
            parsed["_model_confidence"] = model_confidence
            return parsed

        print(
            f"  ⚠️  attempt {attempt}: rows={len(services)} (expected {expected_rows}), "
            f"all_empty={not empty_ok}"
        )
        last_parsed = parsed
        last_model_confidence = model_confidence

    print(f"  ⚠️  All {MAX_RETRIES} attempts exhausted — using last result")
    if last_parsed is not None:
        last_parsed["_model_confidence"] = last_model_confidence
    return last_parsed


# =========================
# MAIN PIPELINE
# =========================
def run_pipeline(pdf_path: str, output_dir: str = "EOB_OUTPUT/Marpai", company_name = "Marpai"):
    # pdf_name is used as eob_id throughout
    pdf_name = os.path.basename(pdf_path).split(".")[0].split("_")[-1]
    pdf_full_name = os.path.basename(pdf_path)

    base_dir          = os.path.join(output_dir, pdf_name)
    cropped_dir       = os.path.join(base_dir, "cropped_images")
    json_output_path  = os.path.join(base_dir, f"{pdf_name}_output.json")

    from output_utils import SUCCESS_DIR, FAILED_DIR
    already_success = os.path.exists(os.path.join(SUCCESS_DIR, company_name, pdf_name, f"{pdf_name}_output.json"))
    already_failed  = os.path.exists(os.path.join(FAILED_DIR, company_name, pdf_name, f"{pdf_name}_output.json"))
    if already_success or already_failed:
        print(f"⏭️  Skipping {pdf_name} — output already exists")
        return None

    os.makedirs(base_dir,    exist_ok=True)
    os.makedirs(cropped_dir, exist_ok=True)

    image_items  = crop_claim_tables(pdf_path, output_dir=cropped_dir)
    is_denied    = check_claim_denied(pdf_path)
    print(f"claim status: {is_denied}")

    final_prompt = build_prompt(pdf_name)

    patients = []

    for idx, item in enumerate(image_items):
        img_path      = item["image_path"]
        expected_rows = item["expected_rows"]

        print(f"\nProcessing table {idx+1}/{len(image_items)}")

        image_cv  = make_table(img_path)
        image_pil = Image.fromarray(image_cv).convert("RGB")

        parsed = run_model_with_retry(image_pil, final_prompt, expected_rows)

        if parsed is None:
            print(f"❌ Skipping table {idx+1} — model returned nothing usable")
            continue

        parsed["eob_id"] = pdf_name
        parsed = convert_amounts_to_string(parsed)
        parsed["_expected_rows"] = expected_rows

        # Belt-and-suspenders: normalize service_date even if the model
        # returned the full range instead of just the end date.
        for svc in parsed.get("services", []):
            svc["service_date"] = normalize_service_date(svc.get("service_date", ""))

        print("Extracted:")
        print(json.dumps(parsed, indent=2))

        date_of_service = ""
        if parsed.get("services"):
            date_of_service = parsed["services"][0].get("service_date", "")

        patient_data = {
            "claimant":          parsed.get("claimant", ""),
            "provider_name":    parsed.get("provider_name", ""),
            #"date_of_service":  date_of_service,
            "services":         parsed.get("services", []),
            "totals":           parsed.get("totals", {}),
            "_expected_rows":   expected_rows,
            "_model_confidence": parsed.get("_model_confidence"),
        }
        patients.append(patient_data)

    # Validate every patient/claim block
    for patient in patients:
        is_valid, log, errors = validate_patient_totals(
            patient=patient,
            patient_name=patient.get("claimant", "UNKNOWN"),
            expected_row_count=patient.get("_expected_rows", 0),
        )
        patient["validation"] = {
            "status": is_valid,
            "errors": errors,
        }
        patient.pop("_expected_rows", None)

    # Overall document confidence, computed from each patient's
    # per-field model confidence before we strip the internal keys.
    confidence_results = list(patients)
    confidence_score = calculate_eob_confidence(confidence_results)

    for patient in patients:
        patient.pop("_model_confidence", None)

        

    final = [
        {
            "eob_id":       pdf_name,
            "file_name":pdf_full_name,
            "claim_status": is_denied,
            "payor": "Marpai Health",
            "confidence_score": confidence_score,
            "Generated_CDT_code": True,
            "patients":     patients,
        }
    ]

    success_path, failed_path = save_split_output(
                                final,
                                company_name=company_name,
                                pdf_name=pdf_name,
                                pdf_path=pdf_path,
                                cropped_dir=cropped_dir,
                            )
                        
    print(f"\n📁 Cropped images : {cropped_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final